# THÍ NGHIỆM KIỂM CHỨNG: RELEVANCE GATE DIAGNOSTIC (Stanceosaurus)

**Mục đích**: Kiểm tra giả thuyết nền của hướng "tách Relevance ra khỏi Stance" (ArguGraph / Relevance Gate) TRƯỚC khi đầu tư xây dựng kiến trúc đồ thị phức tạp.

**Câu hỏi thí nghiệm**: Nếu ta huấn luyện MỘT classifier nhị phân chuyên biệt chỉ để trả lời "utterance này có phải lớp Other (lạc đề/không rõ lập trường) hay không" (bỏ qua việc phân biệt Supporting vs Refuting), thì F1 của lớp Other có cải thiện rõ rệt so với khi phải phân loại đồng thời 3 lớp (SITPCL-FoT: 40.45, BERTweet baseline 3-way: 67.71, SITPCL gốc: 60.13) không?

**Logic quyết định**:
- Nếu F1(Other, binary task) **không** vượt rõ rệt mức 60-67 (ngang hoặc thấp hơn baseline 3-way) → bài toán khó không nằm ở việc phải chia 3 lớp cùng lúc, mà nằm ở chính nội dung (lạc đề mỉa mai trông giống có lập trường) → giả thuyết "tách relevance ra riêng sẽ dễ hơn" SAI → loại bỏ hướng ArguGraph/Relevance-Gate.
- Nếu F1 tăng rõ rệt (vượt hẳn 67.71) → giả thuyết có cơ sở, nên đầu tư tiếp vào Relevance Gate (không cần đồ thị lan truyền).

**Thiết kế**: Backbone BERTweet giống các thí nghiệm trước (đồng bộ để so sánh công bằng), KHÔNG dùng SSE/FoT/Sparsemax, KHÔNG dùng cấu trúc đồ thị — chỉ là 1 classifier nhị phân thuần content, đúng kiểu baseline BERTweet nhưng đổi bài toán từ 3-way sang binary "Other vs Non-Other".

In [ ]:
# ==============================================================================
# STEP 1: CAI DAT THU VIEN & TU DONG DO TIM DUONG DAN DU LIEU
# ==============================================================================
!pip install -q transformers scikit-learn

import os
import glob
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, f1_score, precision_recall_fscore_support
from tqdm.auto import tqdm

found_files = glob.glob('/kaggle/input/**/train_conversations.json', recursive=True) + \
              glob.glob('./**/train_conversations.json', recursive=True) + \
              glob.glob('D:/KLTN/**/train_conversations.json', recursive=True)

if not found_files:
    raise FileNotFoundError('Khong tim thay train_conversations.json! Kiem tra Input Dataset tren Kaggle.')

DATA_DIR = os.path.dirname(found_files[0])
print(f'--> Da ket noi du lieu Stanceosaurus Reconstructed tai: {DATA_DIR}')

In [ ]:
# ==============================================================================
# STEP 2: CAU HINH
# ==============================================================================
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

class Config:
    data_dir = DATA_DIR
    model_name = 'vinai/bertweet-base'
    max_seq_len = 128  # BERTweet-base chi pretrain voi seq_len=128, vuot qua se gay
                        # CUDA device-side assert (index out of range o position embedding)
    batch_size = 16
    num_epochs = 4
    learning_rate = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.1
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = Config()
print('Device:', config.device)

In [ ]:
# ==============================================================================
# STEP 3: NAP DU LIEU & XAY LAI NHAN NHI PHAN "IS_OTHER"
# label goc: 0=Supporting, 1=Refuting, 2=Other -> is_other = 1 if label==2 else 0
# Dung lai dung ranh gioi lop Other de so sanh cong bang voi F1_Other cua cac
# thi nghiem truoc (SITPCL=60.13, BERTweet baseline=67.71, SITPCL-FoT=40.45)
# ==============================================================================
def load_dialogue_json(filename):
    file_path = os.path.join(config.data_dir, filename)
    with open(file_path, 'r', encoding='utf-8') as f:
        samples = json.load(f)

    rows = []
    for s in samples:
        sentences = s['sentences']
        speakers = s['speakers']
        target = str(s['target']).strip()
        label = int(s['label'])
        is_other = 1 if label == 2 else 0

        # Noi toan bo context hoi thoai (toi da 6 luot da duoc sliding-window tu truoc)
        # thanh 1 doan van co danh dau nguoi noi, giong cach BERTweet baseline xu ly.
        turns = [f"User{speakers[i]}: {sentences[i]}" for i in range(len(sentences))]
        context_text = ' </s> '.join(turns)

        rows.append({
            'target': target,
            'context_text': context_text,
            'is_other': is_other,
            'label_3way': label,
        })
    return pd.DataFrame(rows)

print('[*] Dang nap du lieu...')
train_df = load_dialogue_json('train_conversations.json')
dev_df = load_dialogue_json('dev_conversations.json')
test_df = load_dialogue_json('test_conversations.json')

for name, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
    print(f'{name}: {len(df)} mau, ty le is_other={df.is_other.mean():.3f}')

In [ ]:
# ==============================================================================
# STEP 4: TOKENIZATION & DATASET
# ==============================================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=False)

# Bao ve an toan: khong bao gio vuot qua model_max_length cua chinh tokenizer/model
# (nguyen nhan goc cua loi CUDA device-side assert truoc do).
safe_max_len = min(config.max_seq_len, getattr(tokenizer, 'model_max_length', config.max_seq_len))
if safe_max_len != config.max_seq_len:
    print(f'[!] max_seq_len bi gioi han tu {config.max_seq_len} xuong {safe_max_len} theo model_max_length cua tokenizer.')
config.max_seq_len = safe_max_len

class RelevanceDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.data = df.to_dict('records')
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # Prompt don gian: context hoi thoai + target, khong dung speaker-interaction module
        text = f"{item['context_text']} </s> Target: {item['target']}"
        enc = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label': torch.tensor(item['is_other'], dtype=torch.long),
        }

train_loader = DataLoader(RelevanceDataset(train_df, tokenizer, config.max_seq_len), batch_size=config.batch_size, shuffle=True)
dev_loader = DataLoader(RelevanceDataset(dev_df, tokenizer, config.max_seq_len), batch_size=config.batch_size, shuffle=False)
test_loader = DataLoader(RelevanceDataset(test_df, tokenizer, config.max_seq_len), batch_size=config.batch_size, shuffle=False)
print('So batch train/dev/test:', len(train_loader), len(dev_loader), len(test_loader))

In [ ]:
# ==============================================================================
# STEP 5: MODEL - CHI 1 BERTWEET ENCODER + 1 LINEAR HEAD NHI PHAN
# Khong SSE, khong FoT, khong Sparsemax -- co tinh la bai test "co ban nhat co the"
# cho gia thuyet relevance-detection.
# ==============================================================================
class RelevanceClassifier(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 2)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]  # token dau (tuong duong [CLS]/<s>)
        logits = self.classifier(self.dropout(pooled))
        return logits

model = RelevanceClassifier(config.model_name).to(config.device)
print('So tham so:', sum(p.numel() for p in model.parameters()))

In [ ]:
# ==============================================================================
# STEP 6: TRAINING LOOP
# ==============================================================================
optimizer = AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
total_steps = len(train_loader) * config.num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * config.warmup_ratio), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()

def evaluate(loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(config.device)
            attention_mask = batch['attention_mask'].to(config.device)
            labels = batch['label'].to(config.device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    f1_other = f1_score(all_labels, all_preds, pos_label=1, average='binary', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return f1_other, f1_macro, all_preds, all_labels

best_dev_f1 = -1.0
best_state = None

for epoch in range(config.num_epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.num_epochs}')
    for batch in pbar:
        input_ids = batch['input_ids'].to(config.device)
        attention_mask = batch['attention_mask'].to(config.device)
        labels = batch['label'].to(config.device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    dev_f1_other, dev_f1_macro, _, _ = evaluate(dev_loader)
    print(f'[Epoch {epoch+1}] train_loss={total_loss/len(train_loader):.4f} | dev F1(Other)={dev_f1_other*100:.2f} | dev F1(macro)={dev_f1_macro*100:.2f}')

    if dev_f1_other > best_dev_f1:
        best_dev_f1 = dev_f1_other
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  --> Checkpoint moi tot nhat (dev F1 Other = {best_dev_f1*100:.2f})')

model.load_state_dict(best_state)
print('\nDa nap lai checkpoint tot nhat theo dev F1(Other).')

In [ ]:
# ==============================================================================
# STEP 7: DANH GIA TREN TEST & KET LUAN SO VOI CAC MOC THAM CHIEU
# ==============================================================================
test_f1_other, test_f1_macro, test_preds, test_labels = evaluate(test_loader)

print('='*80)
print('KET QUA THI NGHIEM RELEVANCE-GATE DIAGNOSTIC (binary: Other vs Non-Other)')
print('='*80)
print(classification_report(test_labels, test_preds, target_names=['Non-Other(Supporting/Refuting)', 'Other'], digits=4))
print(f'F1(Other) binary task = {test_f1_other*100:.2f}')
print(f'F1 macro (2 lop)      = {test_f1_macro*100:.2f}')

print('\n--- SO SANH VOI F1(Other) tu cac thi nghiem 3-way truoc do ---')
reference = {
    'SITPCL goc (3-way)': 60.13,
    'BERTweet baseline (3-way)': 67.71,
    'SITPCL + Sparsemax + FoT (3-way)': 40.45,
    'Relevance-Gate Diagnostic (binary, thi nghiem nay)': round(test_f1_other*100, 2),
}
for k, v in reference.items():
    print(f'  {k:50s}: {v}')

diagnostic_f1 = reference['Relevance-Gate Diagnostic (binary, thi nghiem nay)']
baseline_3way = reference['BERTweet baseline (3-way)']

print('\n--- KET LUAN TU DONG (nguong tham khao: vuot ro ret baseline 3-way nghia la >= +3 diem) ---')
if diagnostic_f1 >= baseline_3way + 3:
    print(f'=> F1 tang ro ret ({diagnostic_f1} vs {baseline_3way}). Gia thuyet relevance-gate CO CO SO.')
    print('   Nen dau tu tiep vao Relevance Gate (KHONG can graph propagation/ArguGraph).')
elif diagnostic_f1 <= baseline_3way - 3:
    print(f'=> F1 THAP HON han baseline 3-way ({diagnostic_f1} vs {baseline_3way}).')
    print('   Gia thuyet SAI: tach relevance ra rieng khong de hon, co the con kho hon.')
    print('   NEN LOAI BO huong ArguGraph/Relevance-Gate, quay lai phan tich domain/data khac.')
else:
    print(f'=> F1 xap xi baseline 3-way ({diagnostic_f1} vs {baseline_3way}), khong co khac biet ro ret.')
    print('   Gia thuyet KHONG duoc cung co: viec phai phan 3 lop dong thoi KHONG phai nguyen nhan chinh')
    print('   lam SITPCL+FoT te tren lop Other. Loai bo huong Relevance-Gate/ArguGraph thuan.')